In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if needed for your folder structure.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "BPQ_J.xpt"
csv_file = DATA_DIR / "BPQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2017-2018 BPQ_J XPT -> CSV")
print("=" * 100)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BPQ020",
    "BPQ030",
    "BPD035",
    "BPQ040A",
    "BPQ050A",
    "BPQ080",
    "BPQ060",
    "BPQ070",
    "BPQ090D",
    "BPQ100D"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BPQ_J columns/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6161:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 11:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,161 rows x 11 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached BPQ_J:
#
#     character fields          = NONE
#     genuine decimal fields    = NONE
#
#
# All released values are:
#
#     SEQN identifier
#     categorical questionnaire codes
#     age in whole years
#
#
# Therefore all 11 variables can be converted to
# nullable Int64 AFTER proving every populated value
# is mathematically whole-number.
#
#
# IMPORTANT:
#
# This conclusion applies ONLY to BPQ_J.
#
# It does NOT change the overall NHANES fidelity rule:
#
#     genuine Dietary decimals
#     genuine Lab decimals
#     genuine Examination decimals
#     survey/sample weights
#
# remain exactly AS IS.
#

STRING_COLUMNS = []

DECIMAL_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 11:

    raise ValueError(
        "Expected exactly 11 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Character fields:",
    STRING_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# Direct inspection of BPQ_J:
#
#     tiny artifacts = 0
#
# Therefore NO tiny -> zero correction is required.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 11 BPQ_J variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


ordinary_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)

print(
    "Ordinary numeric zeros:",
    f"{ordinary_zero_total:,}"
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in BPQ_J."
    )


if ordinary_zero_total != 0:

    print(
        ordinary_zero_counts[
            ordinary_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary numeric zeros "
        "found in BPQ_J."
    )


print(
    "PASS: BPQ_J contains 0 tiny artifacts "
    "and 0 ordinary numeric zeros."
)


# ============================================================
# 6. NO ZERO RESTORATION REQUIRED
# ============================================================

#
# Since there is no zero corruption,
# source_df itself is our exact numerical reference.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-to-zero repair required."
)


# ============================================================
# 7. DETECT GENUINE DECIMAL / FRACTIONAL VALUES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is ONLY used diagnostically.
#
# It NEVER overwrites source data.
#

print(
    "\n--- FRACTIONAL VALUE DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = count


    print(
        f"{col}: "
        f"{count:,} fractional observations"
    )


    if count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal fields found.\n\n"
        "These values will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "\nPASS: No genuine fractional/decimal "
    "variables found in BPQ_J."
)


# ============================================================
# 8. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print(
    "\n--- SEQN BEFORE CONVERSION ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93705:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 93705-102956 validated."
)


# ============================================================
# 9. PROVE ALL 11 FIELDS ARE WHOLE-NUMBER
# ============================================================

whole_number_failures = []


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        whole_number_failures.append(
            col
        )


if whole_number_failures:

    raise ValueError(
        "Unexpected fractional values found in:\n"
        + str(
            whole_number_failures
        )
    )


print(
    "PASS: All 11 BPQ_J variables are "
    "mathematically whole-number."
)


# ============================================================
# 10. CONVERT ALL 11 FIELDS TO NULLABLE Int64
# ============================================================

#
# NO .round() is used.
#
# Example:
#
#     93705.0 -> 93705
#
# is formatting cleanup only.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All 11 variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 11. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - BPQ020
# ============================================================

#
# Ever told had high blood pressure:
#
#     1 = Yes          2,137
#     2 = No           4,014
#     7 = Refused          0
#     9 = Don't know      10
#
# Missing = 0
#

check_frequency(
    "BPQ020",
    {
        1: 2137,
        2: 4014,
        9: 10
    },
    0
)


# ============================================================
# 13. CDC CHECK - BPQ030
# ============================================================

#
# Told had hypertension on 2+ visits:
#
#     1 = Yes          1,718
#     2 = No             408
#     7 = Refused          0
#     9 = Don't know      11
#
# Missing = 4,024
#

check_frequency(
    "BPQ030",
    {
        1: 1718,
        2: 408,
        9: 11
    },
    4024
)


# ============================================================
# 14. BPQ020 -> BPQ030 ROUTING
# ============================================================

bpq020_yes = (
    df["BPQ020"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq030_present = (
    df["BPQ030"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq030_routing_mismatch = int(
    np.sum(
        bpq020_yes
        !=
        bpq030_present
    )
)


print(
    "\n--- BPQ020 / BPQ030 ROUTING ---"
)

print(
    "BPQ020=1:",
    int(
        bpq020_yes.sum()
    )
)

print(
    "BPQ030 populated:",
    int(
        bpq030_present.sum()
    )
)

print(
    "Routing mismatches:",
    bpq030_routing_mismatch
)


if int(
    bpq020_yes.sum()
) != 2137:

    raise ValueError(
        "Unexpected BPQ020=1 count."
    )


if bpq030_routing_mismatch != 0:

    raise ValueError(
        "BPQ030 availability differs "
        "from BPQ020=1 routing."
    )


print(
    "PASS: BPQ020/BPQ030 routing validated."
)


# ============================================================
# 15. CDC CHECK - BPD035
# ============================================================

#
# Age first told had hypertension.
#
# CDC disclosure editing:
#
#     <12 -> released as 12
#     >80 -> released as 80
#
#
# Frequencies:
#
#     13-79 = 2,036
#     12    =    11
#     80    =    24
#     777   =     0
#     999   =    66
#
# Missing = 4,024
#

bpd035_range = int(
    df["BPD035"]
    .between(
        13,
        79
    )
    .sum()
)


bpd035_12 = int(
    df["BPD035"]
    .eq(12)
    .sum()
)


bpd035_80 = int(
    df["BPD035"]
    .eq(80)
    .sum()
)


bpd035_777 = int(
    df["BPD035"]
    .eq(777)
    .sum()
)


bpd035_999 = int(
    df["BPD035"]
    .eq(999)
    .sum()
)


bpd035_missing = int(
    df["BPD035"]
    .isna()
    .sum()
)


print(
    "\n--- BPD035 ---"
)

print(
    "13-79:",
    f"{bpd035_range:,}"
)

print(
    "12 or younger:",
    bpd035_12
)

print(
    "80 or older:",
    bpd035_80
)

print(
    "777:",
    bpd035_777
)

print(
    "999:",
    bpd035_999
)

print(
    "Missing:",
    f"{bpd035_missing:,}"
)


if bpd035_range != 2036:

    raise ValueError(
        "BPD035 age 13-79 count differs."
    )


if bpd035_12 != 11:

    raise ValueError(
        "BPD035 age <=12 top/bottom code differs."
    )


if bpd035_80 != 24:

    raise ValueError(
        "BPD035 age >=80 top-code count differs."
    )


if bpd035_777 != 0:

    raise ValueError(
        "Unexpected BPD035 refused values."
    )


if bpd035_999 != 66:

    raise ValueError(
        "BPD035 don't-know count differs."
    )


if bpd035_missing != 4024:

    raise ValueError(
        "BPD035 missing count differs."
    )


print(
    "PASS: BPD035 matches CDC."
)


# ============================================================
# 16. BPQ020 -> BPD035 ROUTING
# ============================================================

bpd035_present = (
    df["BPD035"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpd035_routing_mismatch = int(
    np.sum(
        bpq020_yes
        !=
        bpd035_present
    )
)


if bpd035_routing_mismatch != 0:

    raise ValueError(
        "BPD035 availability differs "
        "from BPQ020=1."
    )


print(
    "PASS: BPQ020/BPD035 routing validated."
)


# ============================================================
# 17. CDC CHECK - BPQ040A
# ============================================================

check_frequency(
    "BPQ040A",
    {
        1: 1942,
        2: 192,
        7: 1,
        9: 2
    },
    4024
)


# ============================================================
# 18. BPQ020 -> BPQ040A ROUTING
# ============================================================

bpq040a_present = (
    df["BPQ040A"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq040a_routing_mismatch = int(
    np.sum(
        bpq020_yes
        !=
        bpq040a_present
    )
)


if bpq040a_routing_mismatch != 0:

    raise ValueError(
        "BPQ040A availability differs "
        "from BPQ020=1."
    )


print(
    "PASS: BPQ020/BPQ040A routing validated."
)


# ============================================================
# 19. CDC CHECK - BPQ050A
# ============================================================

check_frequency(
    "BPQ050A",
    {
        1: 1650,
        2: 292
    },
    4219
)


# ============================================================
# 20. BPQ040A -> BPQ050A ROUTING
# ============================================================

bpq050a_expected = (
    df["BPQ040A"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq050a_present = (
    df["BPQ050A"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq050a_routing_mismatch = int(
    np.sum(
        bpq050a_expected
        !=
        bpq050a_present
    )
)


print(
    "\n--- BPQ040A / BPQ050A ROUTING ---"
)

print(
    "BPQ040A=1:",
    int(
        bpq050a_expected.sum()
    )
)

print(
    "BPQ050A populated:",
    int(
        bpq050a_present.sum()
    )
)

print(
    "Routing mismatches:",
    bpq050a_routing_mismatch
)


if int(
    bpq050a_expected.sum()
) != 1942:

    raise ValueError(
        "Unexpected BPQ050A eligibility count."
    )


if bpq050a_routing_mismatch != 0:

    raise ValueError(
        "BPQ050A availability differs "
        "from BPQ040A=1."
    )


print(
    "PASS: BPQ040A/BPQ050A routing validated."
)


# ============================================================
# 21. CDC CHECK - BPQ080
# ============================================================

check_frequency(
    "BPQ080",
    {
        1: 1968,
        2: 4153,
        7: 2,
        9: 38
    },
    0
)


# ============================================================
# 22. CDC CHECK - BPQ060
# ============================================================

check_frequency(
    "BPQ060",
    {
        1: 2575,
        2: 1411,
        7: 1,
        9: 206
    },
    1968
)


# ============================================================
# 23. BPQ080 -> BPQ060 ROUTING
# ============================================================

#
# CDC:
#
# BPQ080=1 skips BPQ060 and goes directly to BPQ070.
#
# BPQ060 therefore appears for:
#
#     BPQ080 = 2, 7, or 9
#

bpq060_expected = (
    df["BPQ080"]
    .isin(
        [2, 7, 9]
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq060_present = (
    df["BPQ060"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq060_routing_mismatch = int(
    np.sum(
        bpq060_expected
        !=
        bpq060_present
    )
)


print(
    "\n--- BPQ080 / BPQ060 ROUTING ---"
)

print(
    "Expected BPQ060:",
    int(
        bpq060_expected.sum()
    )
)

print(
    "BPQ060 populated:",
    int(
        bpq060_present.sum()
    )
)

print(
    "Routing mismatches:",
    bpq060_routing_mismatch
)


if int(
    bpq060_expected.sum()
) != 4193:

    raise ValueError(
        "Unexpected BPQ060 eligibility count."
    )


if bpq060_routing_mismatch != 0:

    raise ValueError(
        "BPQ060 availability differs "
        "from BPQ080 routing."
    )


print(
    "PASS: BPQ080/BPQ060 routing validated."
)


# ============================================================
# 24. CDC CHECK - BPQ070
# ============================================================

check_frequency(
    "BPQ070",
    {
        1: 2779,
        2: 821,
        3: 631,
        4: 298,
        9: 14
    },
    1618
)


# ============================================================
# 25. BPQ070 ELIGIBILITY / ROUTING
# ============================================================

#
# BPQ070 is reached by:
#
#     BPQ080 = 1
#
# OR
#
#     BPQ060 = 1
#
#
# Because BPQ060 is skipped when BPQ080=1,
# these groups do not overlap in the released file.
#

bpq070_expected = (
    (
        df["BPQ080"].eq(1)
        |
        df["BPQ060"].eq(1)
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq070_present = (
    df["BPQ070"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq070_routing_mismatch = int(
    np.sum(
        bpq070_expected
        !=
        bpq070_present
    )
)


print(
    "\n--- BPQ070 ROUTING ---"
)

print(
    "Expected BPQ070:",
    int(
        bpq070_expected.sum()
    )
)

print(
    "BPQ070 populated:",
    int(
        bpq070_present.sum()
    )
)

print(
    "Routing mismatches:",
    bpq070_routing_mismatch
)


if int(
    bpq070_expected.sum()
) != 4543:

    raise ValueError(
        "Unexpected BPQ070 eligibility count."
    )


if bpq070_routing_mismatch != 0:

    raise ValueError(
        "BPQ070 routing differs from CDC flow."
    )


print(
    "PASS: BPQ070 routing validated."
)


# ============================================================
# 26. CDC CHECK - BPQ090D
# ============================================================

check_frequency(
    "BPQ090D",
    {
        1: 1653,
        2: 2877,
        7: 1,
        9: 12
    },
    1618
)


# ============================================================
# 27. BPQ090D ROUTING
# ============================================================

bpq090d_present = (
    df["BPQ090D"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq090d_routing_mismatch = int(
    np.sum(
        bpq070_expected
        !=
        bpq090d_present
    )
)


if bpq090d_routing_mismatch != 0:

    raise ValueError(
        "BPQ090D routing differs "
        "from BPQ070 eligibility."
    )


print(
    "PASS: BPQ090D routing validated."
)


# ============================================================
# 28. CDC CHECK - BPQ100D
# ============================================================

check_frequency(
    "BPQ100D",
    {
        1: 1202,
        2: 445,
        9: 6
    },
    4508
)


# ============================================================
# 29. BPQ090D -> BPQ100D ROUTING
# ============================================================

bpq100d_expected = (
    df["BPQ090D"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq100d_present = (
    df["BPQ100D"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq100d_routing_mismatch = int(
    np.sum(
        bpq100d_expected
        !=
        bpq100d_present
    )
)


print(
    "\n--- BPQ090D / BPQ100D ROUTING ---"
)

print(
    "BPQ090D=1:",
    int(
        bpq100d_expected.sum()
    )
)

print(
    "BPQ100D populated:",
    int(
        bpq100d_present.sum()
    )
)

print(
    "Routing mismatches:",
    bpq100d_routing_mismatch
)


if int(
    bpq100d_expected.sum()
) != 1653:

    raise ValueError(
        "Unexpected BPQ100D eligibility count."
    )


if bpq100d_routing_mismatch != 0:

    raise ValueError(
        "BPQ100D availability differs "
        "from BPQ090D=1."
    )


print(
    "PASS: BPQ090D/BPQ100D routing validated."
)


# ============================================================
# 30. VALID RESPONSE CODE CHECK
# ============================================================

VALID_CODES = {

    "BPQ020": {
        1, 2, 9
    },

    "BPQ030": {
        1, 2, 9
    },

    "BPQ040A": {
        1, 2, 7, 9
    },

    "BPQ050A": {
        1, 2
    },

    "BPQ080": {
        1, 2, 7, 9
    },

    "BPQ060": {
        1, 2, 7, 9
    },

    "BPQ070": {
        1, 2, 3, 4, 9
    },

    "BPQ090D": {
        1, 2, 7, 9
    },

    "BPQ100D": {
        1, 2, 9
    }
}


print(
    "\n--- VALID RESPONSE CODE CHECK ---"
)


for col, allowed_codes in VALID_CODES.items():

    observed_codes = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed_codes
        -
        allowed_codes
    )


    print(
        f"{col}:",
        sorted(
            observed_codes
        )
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )


print(
    "PASS: No unexpected BPQ categorical codes."
)


# ============================================================
# 31. FINAL ZERO CHECK BEFORE EXPORT
# ============================================================

#
# Original XPT:
#
#     tiny zeros    = 0
#     ordinary zero = 0
#
# Processing must not introduce any zero values.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if count > 0:

        final_zero_counts[col] = count


print(
    "\n--- FINAL ZERO CHECK BEFORE EXPORT ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected numeric zeros created:\n"
        + str(
            final_zero_counts
        )
    )


print(
    "PASS: No numeric zeros created."
)


# ============================================================
# 32. NUMERIC FIDELITY BEFORE EXPORT
# ============================================================

#
# BPQ_J requires NO numerical correction.
#
# Therefore every value must remain exactly numerically
# identical to the original XPT.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 11 fields remain numerically "
    "identical to source XPT."
)


# ============================================================
# 33. STORE MISSING COUNTS
# ============================================================

expected_missing_counts = {
    col: int(
        expected_df[col]
        .isna()
        .sum()
    )
    for col in EXPECTED_COLUMNS
}


# ============================================================
# 34. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# DO NOT USE:
#
#     df.round(...)
#
# DO NOT USE:
#
#     float_format=
#
#
# BPQ_J itself contains no genuine decimals.
#
# The same general export rule protects genuine
# decimal variables in your other NHANES datasets.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 35. RE-READ CSV WITH ROUND-TRIP PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 36. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 11 BPQ_J fields."
)


# ============================================================
# 37. MISSING-POSITION FIDELITY
# ============================================================

missing_position_mismatches = 0

missing_count_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_position_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    actual_missing_count = int(
        roundtrip_df[col]
        .isna()
        .sum()
    )


    if (
        actual_missing_count
        !=
        expected_missing_counts[col]
    ):

        missing_count_failures.append(
            col
        )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_position_mismatches
)


if missing_position_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


if missing_count_failures:

    raise ValueError(
        "Missing counts changed in:\n"
        + str(
            missing_count_failures
        )
    )


print(
    "PASS: Missing positions/counts preserved exactly."
)


# ============================================================
# 38. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 39. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 40. RAW CSV .0 CHECK
# ============================================================

#
# BPQ_J has no genuine decimal fields.
#
# Therefore it is safe to check all 11 fields.
#

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


print(
    "\n--- RAW CSV .0 CHECK ---"
)

print(
    dot_zero_errors
)


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any BPQ_J field."
)


# ============================================================
# 41. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected numeric zeros found "
        "in final CSV:\n"
        + str(
            csv_zero_counts
        )
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 42. FINAL KEY FREQUENCY CHECK
# ============================================================

FINAL_FREQUENCIES = {

    "BPQ020": {
        1: 2137,
        2: 4014,
        9: 10
    },

    "BPQ030": {
        1: 1718,
        2: 408,
        9: 11
    },

    "BPQ040A": {
        1: 1942,
        2: 192,
        7: 1,
        9: 2
    },

    "BPQ050A": {
        1: 1650,
        2: 292
    },

    "BPQ080": {
        1: 1968,
        2: 4153,
        7: 2,
        9: 38
    },

    "BPQ060": {
        1: 2575,
        2: 1411,
        7: 1,
        9: 206
    },

    "BPQ070": {
        1: 2779,
        2: 821,
        3: 631,
        4: 298,
        9: 14
    },

    "BPQ090D": {
        1: 1653,
        2: 2877,
        7: 1,
        9: 12
    },

    "BPQ100D": {
        1: 1202,
        2: 445,
        9: 6
    }
}


print(
    "\n--- FINAL FREQUENCY CHECKS ---"
)


for col, expected_freq in FINAL_FREQUENCIES.items():

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    actual_freq = (
        values
        .dropna()
        .astype(int)
        .value_counts()
        .sort_index()
        .to_dict()
    )


    if actual_freq != expected_freq:

        raise ValueError(
            f"Final {col} frequencies changed."
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 43. FINAL BPD035 CHECK
# ============================================================

final_bpd035 = pd.to_numeric(
    raw_csv_df["BPD035"]
    .where(
        raw_csv_df["BPD035"] != "",
        np.nan
    ),
    errors="coerce"
)


if int(
    final_bpd035
    .between(
        13,
        79
    )
    .sum()
) != 2036:

    raise ValueError(
        "Final BPD035 13-79 count changed."
    )


if int(
    final_bpd035
    .eq(12)
    .sum()
) != 11:

    raise ValueError(
        "Final BPD035 age-12 count changed."
    )


if int(
    final_bpd035
    .eq(80)
    .sum()
) != 24:

    raise ValueError(
        "Final BPD035 age-80 count changed."
    )


if int(
    final_bpd035
    .eq(999)
    .sum()
) != 66:

    raise ValueError(
        "Final BPD035 999 count changed."
    )


print(
    "PASS: Final BPD035 distribution preserved."
)


# ============================================================
# 44. FINAL ROUTING CHECKS
# ============================================================

final_bpq020 = pd.to_numeric(
    raw_csv_df["BPQ020"],
    errors="coerce"
)


final_bpq040a = pd.to_numeric(
    raw_csv_df["BPQ040A"]
    .where(
        raw_csv_df["BPQ040A"] != "",
        np.nan
    ),
    errors="coerce"
)


final_bpq080 = pd.to_numeric(
    raw_csv_df["BPQ080"],
    errors="coerce"
)


final_bpq060 = pd.to_numeric(
    raw_csv_df["BPQ060"]
    .where(
        raw_csv_df["BPQ060"] != "",
        np.nan
    ),
    errors="coerce"
)


final_bpq090d = pd.to_numeric(
    raw_csv_df["BPQ090D"]
    .where(
        raw_csv_df["BPQ090D"] != "",
        np.nan
    ),
    errors="coerce"
)


def raw_present(column):

    return (
        raw_csv_df[column]
        .ne("")
        .to_numpy(
            dtype=bool
        )
    )


routing_results = {}


expected = (
    final_bpq020
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


routing_results["BPQ030"] = int(
    np.sum(
        expected
        !=
        raw_present("BPQ030")
    )
)


routing_results["BPD035"] = int(
    np.sum(
        expected
        !=
        raw_present("BPD035")
    )
)


routing_results["BPQ040A"] = int(
    np.sum(
        expected
        !=
        raw_present("BPQ040A")
    )
)


expected = (
    final_bpq040a
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


routing_results["BPQ050A"] = int(
    np.sum(
        expected
        !=
        raw_present("BPQ050A")
    )
)


expected = (
    final_bpq080
    .isin(
        [2, 7, 9]
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


routing_results["BPQ060"] = int(
    np.sum(
        expected
        !=
        raw_present("BPQ060")
    )
)


expected = (
    (
        final_bpq080.eq(1)
        |
        final_bpq060.eq(1)
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


routing_results["BPQ070"] = int(
    np.sum(
        expected
        !=
        raw_present("BPQ070")
    )
)


routing_results["BPQ090D"] = int(
    np.sum(
        expected
        !=
        raw_present("BPQ090D")
    )
)


expected = (
    final_bpq090d
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


routing_results["BPQ100D"] = int(
    np.sum(
        expected
        !=
        raw_present("BPQ100D")
    )
)


print(
    "\n--- FINAL ROUTING CHECKS ---"
)

print(
    routing_results
)


if any(
    count != 0
    for count in routing_results.values()
):

    raise ValueError(
        "One or more final routing checks failed:\n"
        + str(
            routing_results
        )
    )


print(
    "PASS: All 8 final routing checks "
    "have 0 mismatches."
)


# ============================================================
# 45. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 46. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93705:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 47. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "BPQ_J CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2017-2018"
)


print(
    "Component: Questionnaire - "
    "Blood Pressure & Cholesterol"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny XPORT artifacts:",
    tiny_total
)


print(
    "Original ordinary numeric zeros:",
    ordinary_zero_total
)


print(
    "Validated zero restorations:",
    "NONE"
)


print(
    "Genuine decimal variables:",
    fractional_columns
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "BPQ020 hypertension yes:",
    int(
        df["BPQ020"]
        .eq(1)
        .sum()
    )
)


print(
    "BPQ040A advised hypertension medicine:",
    int(
        df["BPQ040A"]
        .eq(1)
        .sum()
    )
)


print(
    "BPQ050A currently taking HBP medicine:",
    int(
        df["BPQ050A"]
        .eq(1)
        .sum()
    )
)


print(
    "BPQ080 high cholesterol yes:",
    int(
        df["BPQ080"]
        .eq(1)
        .sum()
    )
)


print(
    "BPQ090D advised cholesterol medicine:",
    int(
        df["BPQ090D"]
        .eq(1)
        .sum()
    )
)


print(
    "BPQ100D currently taking cholesterol medicine:",
    int(
        df["BPQ100D"]
        .eq(1)
        .sum()
    )
)


print(
    "Routing mismatches:",
    (
        bpq030_routing_mismatch
        +
        bpd035_routing_mismatch
        +
        bpq040a_routing_mismatch
        +
        bpq050a_routing_mismatch
        +
        bpq060_routing_mismatch
        +
        bpq070_routing_mismatch
        +
        bpq090d_routing_mismatch
        +
        bpq100d_routing_mismatch
    )
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "BPQ_J requires NO tiny-zero restoration."
)


print(
    "All 11 released fields are mathematically "
    "whole-number IDs/codes/age values."
)


print(
    "All 11 can therefore be converted to nullable Int64 "
    "without rounding."
)


print(
    "No df.round() and no float_format are used."
)


print(
    "Your broader NHANES fidelity rule remains unchanged:"
)


print(
    "Genuine Dietary and other continuous decimal "
    "values remain exactly AS IS."
)

NHANES 2017-2018 BPQ_J XPT -> CSV
Rows: 6,161
Columns: 11
PASS: Structure = 6,161 rows x 11 columns.

--- FIELD TYPE PLAN ---
Character fields: []
Genuine decimal fields: []
Whole-number/code fields: ['SEQN', 'BPQ020', 'BPQ030', 'BPD035', 'BPQ040A', 'BPQ050A', 'BPQ080', 'BPQ060', 'BPQ070', 'BPQ090D', 'BPQ100D']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Variables containing tiny artifacts: 0
Ordinary numeric zeros: 0
PASS: BPQ_J contains 0 tiny artifacts and 0 ordinary numeric zeros.

PASS: No tiny-to-zero repair required.

--- FRACTIONAL VALUE DETECTION ---
SEQN: 0 fractional observations
BPQ020: 0 fractional observations
BPQ030: 0 fractional observations
BPD035: 0 fractional observations
BPQ040A: 0 fractional observations
BPQ050A: 0 fractional observations
BPQ080: 0 fractional observations
BPQ060: 0 fractional observations
BPQ070: 0 fractional observations
BPQ090D: 0 fractional observations
BPQ100D: 0 fractional observations

PASS: No genuine fractional/decimal variable